> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/Foundations/blob/main/notebooks/05a_Map_Your_County.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Map Your County

**Series:** Map Your County (capstone) (notebook 1 of 1)
**Time:** about 50 minutes

**In this notebook you will:**
- Join a Census table to tract boundaries and check the join
- Clean the data: Census 'no estimate' codes and zero denominators
- Compute a rate and classify it for a choropleth map
- Finish the map with a title, legend, scale bar, north arrow and source, and save it
- Optional: repeat it for your own county

**You should already know:** everything in Foundations notebooks 2a to 4c. This notebook puts it together with less guidance.

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

## The brief

A local food bank wants a **one-page map** of where households without a car live in Tippecanoe County,
Indiana, to plan a delivery service. Your job: produce that map as a PNG file that someone who has never used
a GIS can read.

A finished map answers four questions at a glance: **what** is shown (title and legend with units), **where**
(recognizable shapes, a scale bar and north arrow), **when** and **from what data** (a source line), and
**what is missing** (a "no data" class rather than a misleading zero).

The data:

| File | Contents |
|---|---|
| `tippecanoe_tracts.geojson` | 43 census tract boundaries with `GEOID` |
| `tippecanoe_acs_2024.csv` | ACS 2024 5-year estimates per tract: `total_pop`, `pop_under18`, `median_income`, `poverty_universe`, `below_poverty`, `households`, `hh_no_vehicle` |

Each step below says **what** to do and gives a short hint; try it yourself before opening the solution.

## Step 1: Read both files

Read the tracts with GeoPandas and the table with pandas. Keep `GEOID` as **text** in the table
(`dtype={"GEOID": str}`), as in notebook 4c.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

tracts = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_tracts.geojson")
acs = pd.read_csv("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_acs_2024.csv", dtype={"GEOID": str})
print(len(tracts), "tracts,", len(acs), "table rows")
acs.head()

## Step 2: Join and check

Join the table to the tracts on `GEOID` with `merge()`. Then **check**: the result should still have 43 rows,
and no row should have an empty `total_pop`.

In [ ]:
data = tracts.merge(acs.drop(columns="name"), on="GEOID", how="left")
print(len(data), "rows after the join")
print(data["total_pop"].isna().sum(), "tracts without a match")

`how="left"` keeps every tract even if it has no match, so an unmatched tract shows up as missing values
instead of silently disappearing. Both checks matter: a join can fail by losing rows or by leaving them empty.

## Step 3: Clean the data

Two problems hide in this table. Look at tracts 104 and 105:

In [ ]:
data.loc[data["name"].isin(["Census Tract 104", "Census Tract 105"]),
         ["name", "total_pop", "households", "median_income", "poverty_universe"]]

- `median_income` is **-666666666**: the Census Bureau's code for "no estimate". It must become a missing value
  (`np.nan`), or it will wreck any average or map of income.
- Tract 104 (Purdue's residence halls) has **0 households**: people in dorms live in "group quarters", not
  households. A rate per household cannot be calculated there; it must be missing, **not 0%**.

In [ ]:
data["median_income"] = data["median_income"].where(data["median_income"] >= 0)   # codes become NaN
data.loc[data["name"] == "Census Tract 104", ["median_income"]]

## Step 4: Compute the rate

The percentage of households without a vehicle is `hh_no_vehicle / households * 100`. Use `np.where()` so a
zero denominator gives `np.nan` instead of an error or infinity.

In [ ]:
data["pct_no_vehicle"] = np.where(
    data["households"] > 0,
    data["hh_no_vehicle"] / data["households"] * 100,
    np.nan,
)
data[["name", "households", "hh_no_vehicle", "pct_no_vehicle"]].sort_values(
    "pct_no_vehicle", ascending=False).head(6).round(1)

## Step 5: Project and take a first look

Convert to **EPSG:26916** (UTM zone 16N, meters), so the map is not stretched and the scale bar will be true.
Then make a quick map to see the pattern.

In [ ]:
data = data.to_crs(epsg=26916)
data.plot(column="pct_no_vehicle", legend=True, missing_kwds={"color": "lightgray"})
plt.show()

## Step 6: Choose classes

A choropleth needs a small number of classes. Look at the distribution first: it is **skewed**, with most
tracts below 10% and a few far above. Equal intervals would put almost every tract in the lowest class, so
**natural breaks** or **quantiles** suit it better. The Data Visualization course's
[Fundamentals of Map Design](https://www.cyberfaces.org/learn/module/441-fundamentals-of-map-design) module compares the
methods. Here we use `FisherJenks`, mapclassify's natural breaks method, which gives the same classes
every time it runs (QGIS calls this idea **Natural Breaks (Jenks)**).

In [ ]:
plt.hist(data["pct_no_vehicle"].dropna(), bins=15, color="tab:blue")
plt.xlabel("Households without a vehicle (%)")
plt.ylabel("Number of tracts")
plt.show()

import mapclassify
print(mapclassify.FisherJenks(data["pct_no_vehicle"].dropna(), k=5))

## Step 7: The finished map

Everything a reader needs, in one figure. The two small helper functions draw a **scale bar** and a **north
arrow**; they work because the map is in meters.

In [ ]:
def add_scale_bar(ax, length_m=10000, label="10 km"):
    """Draw a simple scale bar in the lower right. The map must be in a CRS measured in meters."""
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    x = x1 - (x1 - x0) * 0.05 - length_m
    y = y0 + (y1 - y0) * 0.05
    ax.plot([x, x + length_m], [y, y], color="black", linewidth=3)
    ax.text(x + length_m / 2, y + (y1 - y0) * 0.015, label, ha="center", va="bottom", fontsize=9)


def add_north_arrow(ax):
    ax.annotate("N", xy=(0.06, 0.95), xytext=(0.06, 0.85), xycoords="axes fraction",
                ha="center", va="center", fontsize=12, fontweight="bold",
                arrowprops=dict(facecolor="black", width=4, headwidth=10))


fig, ax = plt.subplots(figsize=(10, 8))
data.plot(
    ax=ax,
    column="pct_no_vehicle",
    scheme="FisherJenks", k=5,
    cmap="YlOrRd",
    edgecolor="white", linewidth=0.5,
    legend=True,
    legend_kwds={"title": "Households without\na vehicle (%)", "loc": "upper left", "bbox_to_anchor": (1.0, 0.9),
                 "fmt": "{:.0f}"},
    missing_kwds={"color": "lightgray", "hatch": "///", "edgecolor": "white", "label": "No households (dorms)"},
)
ax.set_title("Where households without a car live in Tippecanoe County, Indiana",
             fontsize=13, fontweight="bold", loc="left")
add_scale_bar(ax)
add_north_arrow(ax)
ax.set_axis_off()
fig.text(0.02, 0.02, "Data: U.S. Census Bureau, American Community Survey 2020-2024 5-year estimates, "
         "census tracts. Map by A. Student, 2026.", fontsize=8, color="dimgray")
fig.savefig("tippecanoe_no_vehicle.png", dpi=200, bbox_inches="tight")
plt.show()

The PNG is saved next to this notebook. To download it, find it in the file browser on the left,
right-click `tippecanoe_no_vehicle.png` and choose **Download**.

## Check your map

Before you share a map, read it as a stranger would:

- Does the **title** say what is shown and where? Could it state the finding instead ("Car-free households
  concentrate near downtown Lafayette and campus")?
- Does the **legend** have units, and readable class labels?
- Is **missing data** visibly different from low values?
- Is there a **source** line with the data's year?
- Would the map still work in **grayscale** or for a reader with color vision deficiency? (`YlOrRd` varies in
  lightness, so it does.)

## Try it yourself

Make a second finished map, of the **poverty rate**: `below_poverty / poverty_universe * 100`.

1. Compute the rate with `np.where()` so tract 104 (poverty universe 0) becomes missing.
2. Copy the map cell, change the column, the legend title, the map title and the file name.
3. Many of the highest-poverty tracts are near Purdue. Add one sentence to the source line that explains why.

*Hint:* The `where` pattern from Step 4 works for any rate.

<details>
<summary><b>Show a solution</b></summary>

```python
data["poverty_rate"] = np.where(data["poverty_universe"] > 0,
                                data["below_poverty"] / data["poverty_universe"] * 100, np.nan)

fig, ax = plt.subplots(figsize=(10, 8))
data.plot(ax=ax, column="poverty_rate", scheme="FisherJenks", k=5, cmap="PuBu",
          edgecolor="white", linewidth=0.5, legend=True,
          legend_kwds={"title": "Below poverty line (%)", "loc": "upper left", "bbox_to_anchor": (1.0, 0.9), "fmt": "{:.0f}"},
          missing_kwds={"color": "lightgray", "hatch": "///", "edgecolor": "white", "label": "Not applicable (dorms)"})
ax.set_title("Poverty rate by census tract, Tippecanoe County, Indiana", fontsize=13, fontweight="bold", loc="left")
add_scale_bar(ax); add_north_arrow(ax); ax.set_axis_off()
fig.text(0.02, 0.02, "Data: U.S. Census Bureau, ACS 2020-2024 5-year estimates. Tracts near Purdue have many "
         "students, who often report low incomes.", fontsize=8, color="dimgray")
fig.savefig("tippecanoe_poverty.png", dpi=200, bbox_inches="tight")
plt.show()
```

</details>

In [ ]:
# Write your answer here

## Optional: map your own county

The boundary files are public and need no key. The ACS table needs your **own free Census API key**
(see the Foundations setup page). Find your state and county FIPS codes (for example on the Census Bureau's
"FIPS codes" pages: Indiana is 18 and Tippecanoe County is 157), fill them in with your key, and run the two
cells. Then rerun Steps 2 to 7 with `tracts` and `acs` replaced. Never share a notebook with your key in it.

In [ ]:
STATE = "18"      # two digits, as text
COUNTY = "157"    # three digits, as text
KEY = "PASTE_YOUR_CENSUS_API_KEY_HERE"

In [ ]:
# Run this cell only after you have filled in your own key above.
if KEY != "PASTE_YOUR_CENSUS_API_KEY_HERE":
    tracts = gpd.read_file(f"https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_{STATE}_tract_500k.zip")
    tracts = tracts[tracts["COUNTYFP"] == COUNTY][["GEOID", "NAMELSAD", "geometry"]].rename(columns={"NAMELSAD": "name"})

    fields = "NAME,B01003_001E,B09001_001E,B19013_001E,B17001_001E,B17001_002E,B25044_001E,B25044_003E,B25044_010E"
    url = (f"https://api.census.gov/data/2024/acs/acs5?get={fields}"
           f"&for=tract:*&in=state:{STATE}%20county:{COUNTY}&key={KEY}")
    raw = pd.read_json(url)
    raw.columns = raw.iloc[0]
    raw = raw[1:]
    acs = pd.DataFrame({
        "GEOID": raw["state"] + raw["county"] + raw["tract"],
        "total_pop": raw["B01003_001E"].astype(int),
        "pop_under18": raw["B09001_001E"].astype(int),
        "median_income": raw["B19013_001E"].astype(int),
        "poverty_universe": raw["B17001_001E"].astype(int),
        "below_poverty": raw["B17001_002E"].astype(int),
        "households": raw["B25044_001E"].astype(int),
        "hh_no_vehicle": raw["B25044_003E"].astype(int) + raw["B25044_010E"].astype(int),
    })
    print(len(tracts), "tracts and", len(acs), "table rows for your county")
else:
    print("Add your key in the cell above first.")

## Recap

- Join a table to boundaries, then check the row count and the unmatched rows
- Replace Census codes such as -666666666 with missing values, and make zero-denominator rates missing, not 0
- Project to meters, classify a skewed rate with natural breaks or quantiles, and show missing data clearly
- A finished map has a title, a legend with units, a scale bar, a north arrow and a source line

**Next:** the QGIS version of this capstone (Map Your County in QGIS), or the Data Processing course's
Geoprocessing in Python episode.